# GAI multiyear stopover model

## Install the ButterflyLifespan package

In [1]:
library(remotes)
remotes::install_github("grosed/ButterflyLifespan/R-package")

Skipping install of 'ButterflyLifespan' from a github remote, the SHA1 (7863d46d) has not changed since last install.
  Use `force = TRUE` to force installation



## Load the library

In [2]:
library(ButterflyLifespan)

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: reshape2



## Load the library

In [3]:
data(dark_green_fritillary_weekly)
dgf_week <- dark_green_fritillary_weekly

## For demonstration purposes, take a random subset of the data

In [4]:
set.seed(1)
dgf_week <- dgf_week %>% filter(SITENO%in%sample(unique(dgf_week$SITENO),30)) %>% filter(YEAR > 2008)

## For demonstration purposes, take a random subset of the data

In [5]:
output <- analysis_multiyear(dgf_week,"weekly","slope")

iter   10 value 1106.258323
iter   20 value 1051.927133
iter   30 value 1021.810554
iter   40 value 1018.738798
iter   50 value 1018.071949
iter   60 value 1017.525706
iter   70 value 1017.276658
iter   80 value 1017.247324
iter   90 value 1017.220785
iter  100 value 1017.216941
iter  110 value 1017.216250
iter  120 value 1017.215029
iter  130 value 1017.208717
iter  140 value 1017.201051
iter  150 value 1017.196108
iter  160 value 1017.194195
iter  170 value 1017.193031
iter  180 value 1017.190595
iter  190 value 1017.188371
iter  200 value 1017.187814
iter  210 value 1017.187753
iter  220 value 1017.187707
iter  230 value 1017.187534
iter  240 value 1017.187420
iter  250 value 1017.187373
iter  260 value 1017.187358
iter  270 value 1017.187357
iter  280 value 1017.187351
iter  290 value 1017.187349
iter  300 value 1017.187348
iter  310 value 1017.187342
iter  320 value 1017.187317
iter  330 value 1017.187304
iter  340 value 1017.187301
iter  350 value 1017.187286
iter  360 value 1017

## For demonstration purposes, take a random subset of the data

In [6]:
post_process <- function(output,setup_level,phi_type){
  trans_phi <- output[[1]][["phi.out"]]
  if (setup_level=="weekly"){
    trans_phi_day <- trans_phi^(1/7)
    ls_phi <- (1/(1-trans_phi_day))
  } else {
    ls_phi <- (1/(1-trans_phi))
  }
  
  Hess = output[[1]][["modelfit"]][["Hessian"]]
  inv.Hess = solve(Hess)
  res = sqrt(diag(inv.Hess))
  
  if (phi_type=="slope"){
    

    phi.cov = scale(1:length(output[[1]][["mu1.est"]]))
    se_phi_int <- exp(output[[1]][["phi.slope"]]*phi.cov)*exp(output[[1]][["phi.int"]])
    se_phi_slope <- phi.cov*exp(output[[1]][["phi.int"]])*exp(output[[1]][["phi.slope"]]*phi.cov)
        
    se_lifespan = vector(mode="numeric",length=length(output[[1]][["mu1.est"]]))
    for (i in 1:length(output[[1]][["mu1.est"]])){
      se_phi_mat <- as.matrix(c(se_phi_int[i],se_phi_slope[i]))
      
      se_phi_row_mat <- t(se_phi_mat)
      
      var_cor_phi <- inv.Hess[(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 1):(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 2),(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 1):(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 2)]
    
      
      se_lifespan[i] <- se_phi_row_mat %*% var_cor_phi %*% se_phi_mat
    }
    
    
    CI_lifespan = 1.96*(sqrt(se_lifespan))
   
    
    plot_df <- data.frame(lifespan=ls_phi,
                          ci_low=ls_phi-CI_lifespan,
                          ci_up=ls_phi+CI_lifespan,
                          year=1:length(output[[1]][["mu1.est"]]))
    
    
  } else if (phi_type=="variable"){
    res=res[(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 1):(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + length(output[[1]][["mu1.est"]]))]
    
    if (setup_level=="daily"){
      SE_ls_d <- (exp(output[[1]][["modelfit"]][["allval"]]))*res
    } else {
      SE_ls_d <- (1/(1-(1/(1+exp(-(output[[1]][["modelfit"]][["allval"]]))))^(1/7))^2)*((1/7)*(1/(1+exp(-(output[[1]][["modelfit"]][["allval"]])))^(-(6/7))))*((exp(-(output[[1]][["modelfit"]][["allval"]])))/(1+exp(-(output[[1]][["modelfit"]][["allval"]])))^2)*(res)
    }
    
    
    plot_df <- data.frame(lifespan=ls_phi,
                          ci_low=ls_phi-(SE_ls_d*1.96),
                          ci_up=ls_phi+(SE_ls_d*1.96),
                          year=1:length(output[[1]][["mu1.est"]]))
    
    
    
  } else if (ohi_type=="doubleslope"){
      phi.cov = scale(1:length(output[[1]][["mu1.est"]]))
      
      se_phi_int <- exp(output[[1]][["phi.slope"]]*phi.cov)*exp(output[[1]][["phi.int"]])*exp(output[[1]][["phi.slope2"]]*phi.cov^2)
      se_phi_slope <- phi.cov*exp(output[[1]][["phi.slope"]]*phi.cov)*exp(output[[1]][["phi.int"]])*exp(output[[1]][["phi.slope2"]]*phi.cov^2)
      se_phi_slope2 <- (2*phi.cov)*phi.cov*exp(output[[1]][["phi.slope"]]*phi.cov)*exp(output[[1]][["phi.int"]])*exp(output[[1]][["phi.slope2"]]*phi.cov^2)

      
      
      se_lifespan = vector(mode="numeric",length=length(output[[1]][["mu1.est"]]))
      for (i in 1:length(output[[1]][["mu1.est"]])){
        se_phi_mat <- as.matrix(c(se_phi_int[i],se_phi_slope[i],se_phi_slope2[i]))
        
        se_phi_row_mat <- t(se_phi_mat)
        
        var_cor_phi <- inv.Hess[(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 1):(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 3),(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 1):(length(output[[1]][["mu1.est"]]) + length(output[[1]][["sigma.out"]]) + 3)]
        
        
        se_lifespan[i] <- se_phi_row_mat %*% var_cor_phi %*% se_phi_mat
      }
      
      
      CI_lifespan = 1.96*(sqrt(se_lifespan))
      
      
      plot_df <- data.frame(lifespan=ls_phi,
                            ci_low=ls_phi-CI_lifespan,
                            ci_up=ls_phi+CI_lifespan,
                            year=1:length(output[[1]][["mu1.est"]]))

  }
  
  return(plot_df)
}

## Post process the output

In [7]:
lifespan <- post_process(output,"weekly","slope")

## Summarise results

In [8]:
head(lifespan)

,lifespan,ci_low,ci_up,year
,<dbl>,<dbl>,<dbl>,<int>
1,6.746041,4.498597,8.993485,1
2,5.214942,3.869772,6.560112,2
3,4.203165,3.117443,5.288887,3
4,3.511534,2.610626,4.412443,4
5,3.022137,2.311480,3.732795,5
6,2.664256,2.129711,3.198801,6
